# Random Forest (Text + Numeric): Predicting `made_it_contract`

Combines TF-IDF features from scout report text (**overview**, **strengths**, **weaknesses**) with numeric scores (**production_score**, **athleticism_score**) using a `FeatureUnion` pipeline.

## 1. Imports

In [ ]:
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
from time import time

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold,
    cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

RANDOM_STATE = 42

## 2. Custom Transformers for FeatureUnion

`FeatureUnion` requires each branch to receive the same input (the full DataFrame) and return a matrix. We use two thin selectors — one that extracts the pre-processed text column, one that extracts the numeric columns.

In [ ]:
class TextSelector(BaseEstimator, TransformerMixin):
    """Extracts a single text column as a list for CountVectorizer."""
    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.column]


class NumericSelector(BaseEstimator, TransformerMixin):
    """Extracts numeric columns as a numpy array."""
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.columns].values

## 3. Load & Prepare Data

In [ ]:
df = pd.read_csv("../data/processed/draft_enriched_with_contracts.csv")

TARGET       = "made_it_contract"
NUMERIC_COLS = ["production_score", "athleticism_score"]

# Drop rows missing the target or either numeric feature
df = df.dropna(subset=[TARGET] + NUMERIC_COLS).copy()
df[TARGET] = df[TARGET].astype(int)

# Combine text columns
df['combined_text'] = (
    df['overview'].fillna('') + ' ' +
    df['strengths'].fillna('') + ' ' +
    df['weaknesses'].fillna('')
).str.strip()

print(f"Rows with text + numeric + target: {len(df)}")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts())
print(f"\nPositive rate: {df[TARGET].mean():.1%}")
df[NUMERIC_COLS].describe()

## 4. Text Pre-processing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def preprocess(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words and len(w) > 1
    ]
    return ' '.join(tokens)

df['text_preproc'] = df['combined_text'].apply(preprocess)

# Drop rows where all text was empty after preprocessing
df = df[df['text_preproc'] != ''].copy()
df = df.reset_index(drop=True)

print(f"Final dataset size: {len(df)}")
print("Sample pre-processed text:")
print(df['text_preproc'].iloc[0])

## 5. Features & Target

In [ ]:
# The pipeline receives the full DataFrame; each branch selects what it needs
X = df[['text_preproc'] + NUMERIC_COLS]
y = df[TARGET].tolist()

print(f"X shape: {X.shape}")
print(f"Positive rate: {np.mean(y):.1%}")

## 6. Combined Pipeline: FeatureUnion → Random Forest

```
                       ┌── TextSelector ── CountVectorizer ── TfidfTransformer ──┐
DataFrame input ───────┤                                                          ├── hstack ── RandomForest
                       └── NumericSelector ── StandardScaler ────────────────────┘
```

In [ ]:
pipeline = Pipeline([
    ('features', FeatureUnion([
        ('text', Pipeline([
            ('selector', TextSelector('text_preproc')),
            ('vect',     CountVectorizer()),
            ('tfidf',    TfidfTransformer()),
        ])),
        ('numeric', Pipeline([
            ('selector', NumericSelector(NUMERIC_COLS)),
            ('scaler',   StandardScaler()),
        ])),
    ])),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE)),
])

parameters = {
    'features__text__vect__max_df':          [0.5, 0.7],
    'features__text__vect__min_df':          [0.005],
    'features__text__vect__ngram_range':     [(1, 1), (1, 2)],
    'features__text__tfidf__use_idf':        [True],
    'features__text__tfidf__norm':           ['l2'],
    'clf__max_depth':                        [10, 20],
    'clf__n_estimators':                     [300],
    'clf__min_samples_leaf':                 [5, 10],
    'clf__class_weight':                     ['balanced'],
}

grid_search = GridSearchCV(
    pipeline, parameters,
    n_jobs=-1, verbose=2,
    scoring='average_precision', cv=3
)

print("Performing grid search...")
pprint(parameters)
t0 = time()
grid_search.fit(X, y)
print(f"\nDone in {time() - t0:.1f}s")
print(f"\nBest PR-AUC (Average Precision): {grid_search.best_score_:.4f}")
print("Best parameters:")
best_params = grid_search.best_estimator_.get_params()
for p in sorted(parameters.keys()):
    print(f"  {p}: {best_params[p]}")

## 7. Evaluation

PR-AUC (Average Precision) is the right metric when we only care about identifying players who made it. A random classifier scores ≈ positive rate.

In [ ]:
best_clf = grid_search.best_estimator_

y_pred_proba = cross_val_predict(
    best_clf, X, y,
    cv=3, method='predict_proba', n_jobs=-1, verbose=2
)[:, 1]

y_pred = (y_pred_proba >= 0.5).astype(int)

baseline = np.mean(y)
pr_auc   = average_precision_score(y, y_pred_proba)

print(classification_report(y, y_pred, target_names=['did not make it', 'made it']))
print(f"PR-AUC (Average Precision): {pr_auc:.4f}  |  Random baseline: {baseline:.4f}")

## 8. 5-Fold Cross-Validated PR-AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_clf, X, y, cv=cv, scoring='average_precision')
print(f"5-fold CV PR-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Fold scores: {cv_scores.round(4)}")

## 9. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y, y_pred,
    display_labels=['Did Not Make It', 'Made It'],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title('Confusion Matrix')

PrecisionRecallDisplay.from_predictions(
    y, y_pred_proba,
    ax=axes[1],
    name='Random Forest (Text + Numeric)'
)
axes[1].axhline(
    y=baseline, color='k', linestyle='--',
    label=f'Random baseline ({baseline:.2f})'
)
axes[1].set_title('Precision-Recall Curve (Made It)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Top Feature Importances

Refit on the full dataset to extract importances. Feature names come from the text vocabulary + the two numeric column names.

In [ ]:
best_clf.fit(X, y)

text_feature_names    = best_clf.named_steps['features'].transformer_list[0][1]\
                            .named_steps['vect'].get_feature_names_out()
numeric_feature_names = np.array(NUMERIC_COLS)
all_feature_names     = np.concatenate([text_feature_names, numeric_feature_names])

importances = best_clf.named_steps['clf'].feature_importances_

top_n   = 30
indices = np.argsort(importances)[::-1][:top_n]

imp_series = pd.Series(importances[indices], index=all_feature_names[indices]).sort_values()

fig, ax = plt.subplots(figsize=(8, 9))
imp_series.plot.barh(ax=ax)
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title(f'Top {top_n} Feature Importances (Text + Numeric RF)')
plt.tight_layout()
plt.show()